In [1]:
# Run this installation command if Gensim is not installed
%pip install gensim

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/24.4 MB 5.0 MB/s eta 0:00:05
   -------- ------------------------------- 5.0/24.4 MB 15.1 MB/s eta 0:00:02
   ------------ --------------------------- 7.3/24.4 MB 13.8 MB/s eta 0:00:02
   --------------- ------------------------ 9.4/24.4 MB 12.4 MB/s eta 0:00:02
   ---------------------- ----------------- 13.9/24.4 MB 14.2 MB/s eta 0:00:01
   --------------------------- ------------ 16.8/24.4 MB 13.9 MB/s eta 0:00:01
   --------------------------------- ------ 20.7/24.4 MB 14.6 MB/s eta 0:00:01
   ---------------------------------------  23.9/24.4 MB 14.6 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 13.1 MB/s  0:00:02

   ---------------------------------------- 0/2 [smart_open]
   --------------

In [2]:
import re
from gensim.models import Word2Vec

In [3]:
#Step 1: Data preparation

text = """
Alice opened the door and found a small passage.
Alice wanted to enter the beautiful garden.
The garden had bright flowers and cool fountains.
Alice loved the flowers in the garden.
Machine learning helps computers understand data.
Machine learning uses words and numerical vectors.
"""

# Convert to lowercase and split into sentences
sentences = re.split(r'[.!?]+', text.lower())

# Define basic stop words
stop_words = {
    "the", "a", "an", "and", "to", "in", "of",
    "had", "was", "is", "it", "into"
}

# Clean and tokenize the sentences
data = []

for sentence in sentences:
    words = re.findall(r'\b[a-z]+\b', sentence)
    words = [word for word in words if word not in stop_words]

    if words:
        data.append(words)

print("Prepared data:")
print(data)

Prepared data:
[['alice', 'opened', 'door', 'found', 'small', 'passage'], ['alice', 'wanted', 'enter', 'beautiful', 'garden'], ['garden', 'bright', 'flowers', 'cool', 'fountains'], ['alice', 'loved', 'flowers', 'garden'], ['machine', 'learning', 'helps', 'computers', 'understand', 'data'], ['machine', 'learning', 'uses', 'words', 'numerical', 'vectors']]


In [4]:
#Step 2: Generate training data

window = 2
training_data = []

for sentence in data:
    for i in range(window, len(sentence) - window):
        context = (
            sentence[i - window:i]
            + sentence[i + 1:i + window + 1]
        )
        target = sentence[i]

        training_data.append((context, target))

print("Training examples:")
for context, target in training_data[:5]:
    print("Context:", context, "-> Target:", target)

Training examples:
Context: ['alice', 'opened', 'found', 'small'] -> Target: door
Context: ['opened', 'door', 'small', 'passage'] -> Target: found
Context: ['alice', 'wanted', 'beautiful', 'garden'] -> Target: enter
Context: ['garden', 'bright', 'cool', 'fountains'] -> Target: flowers
Context: ['machine', 'learning', 'computers', 'understand'] -> Target: helps


In [5]:
#Step 3: Train the CBOW model

model = Word2Vec(
    sentences=data,
    vector_size=50,
    window=2,
    min_count=1,
    sg=0,
    workers=1,
    seed=42,
    epochs=100
)

print("CBOW model trained successfully!")

CBOW model trained successfully!


In [6]:
# Step 4: Display the output

# Display the vocabulary
print("Vocabulary:")
print(model.wv.index_to_key)

# Display the vector for a word
word = "garden"

print("\nVector for", word, ":")
print(model.wv[word])

# Display similar words
print("\nWords similar to", word, ":")
for similar_word, score in model.wv.most_similar(word, topn=5):
    print(similar_word, ":", round(score, 3))

Vocabulary:
['garden', 'alice', 'learning', 'machine', 'flowers', 'vectors', 'numerical', 'words', 'uses', 'data', 'understand', 'computers', 'helps', 'loved', 'fountains', 'cool', 'bright', 'beautiful', 'enter', 'wanted', 'passage', 'small', 'found', 'door', 'opened']

Vector for garden :
[-0.01633274  0.01102503  0.00625744 -0.00221687 -0.00278253  0.0146807
 -0.01697532  0.00808927 -0.01209364 -0.01622651  0.0014542   0.01970554
  0.00952797  0.01088101  0.00900443  0.01166966  0.00047059 -0.01547467
  0.01336036 -0.00202523 -0.00022629 -0.00544004 -0.01281134  0.01693764
  0.0118253   0.00535269 -0.00381988  0.01264465  0.00183305 -0.00235925
 -0.0017639  -0.0107041  -0.01659661  0.00233676  0.01554961 -0.01737106
  0.01425784  0.01283835 -0.00909281  0.00538599 -0.01323572  0.01055395
  0.00828663 -0.00588014 -0.01692213  0.01853823 -0.00221419  0.01587088
  0.00735575  0.01077091]

Words similar to garden :
bright : 0.249
understand : 0.139
wanted : 0.131
found : 0.094
numerical 